# Phase-0B — UltraFedFM Second-USFM External Transfer Replication v1.0

## Purpose

Phase-0A found a strong external advantage for URFM over an architecture-matched ImageNet-MAE
ViT-B/16 baseline.

This notebook asks one narrow replication question:

> **Does a second ultrasound-specific ViT-B/16 foundation model also outperform the same generic
ImageNet-MAE representation under the same reciprocal breast-ultrasound external-shift protocol?**

This is the last replication gate before investing in the full paper.

---

## Model

### UltraFedFM
Official model:
**UltraFedFM — From pretraining to privacy: federated ultrasound foundation model with
self-supervised learning**

Published encoder:
- ViT-Base
- patch size 16
- 12 Transformer blocks
- embedding dimension 768
- 224×224 input

The official repository provides a pretrained checkpoint through Google Drive / OneDrive / Baidu.

### Frozen feature protocol

For comparability with Phase-0A:
- 224×224 square resize
- ImageNet normalization
- CLS-token output
- frozen encoder
- 768-dimensional representation
- identical linear-head training

**Important:** UltraFedFM's official supervised downstream script defaults to global average
pooling. Phase-0B deliberately uses the same CLS-token extraction rule as URFM and ImageNet-MAE so
the experiment compares pretraining representations under one common frozen readout protocol.
This is not a reproduction of UltraFedFM's published fine-tuning pipeline.

---

## Provenance boundary

The UltraFedFM paper explicitly uses BUSI/BUS-BRA/BUS-UCLM in downstream/fairness experiments.
The released checkpoint used here must be the **pretrained checkpoint**, not any task-finetuned
checkpoint.

The main article states that pretraining used 1,015,754 unlabeled ultrasound images, of which
782,513 were public. Exact public pretraining membership is documented in Supplementary Table 1.

Until that table is independently reconciled with our three evaluation datasets:

- UltraFedFM is a **supportive second-USFM replication**;
- it is not used to claim proven zero pretraining overlap;
- the primary declared-provenance external result remains Phase-0A URFM vs MAE.

This provenance label is frozen before seeing Phase-0B results.

---

## Reused artifacts

No ImageNet-MAE backbone inference or MAE head training is repeated.

Reused from Phase-0A:
- ImageNet-MAE BUSI / BUS-BRA / BrEaST feature caches
- ImageNet-MAE source-trained heads for seeds 42/43/44
- exact BUS-BRA and BrEaST source splits
- Phase-0A URFM-vs-MAE external effect table

Only UltraFedFM:
- checkpoint load
- three dataset feature caches
- 2 sources × 3 head seeds

are new.

---

## Sources and targets

Same Phase-0A matrix:

1. BUS-BRA source → BrEaST
2. BUS-BRA source → BUSI
3. BrEaST source → BUS-BRA
4. BrEaST source → BUSI

BUS-BRA evaluation is patient-level.
BrEaST evaluation is case-level.
BUSI is image-level because reliable patient IDs are unavailable.

---

## Linear-head protocol

For UltraFedFM:
- seeds = 42,43,44
- AdamW
- lr = 1e-4
- weight decay = 1e-2
- batch = 16
- class-weighted CE
- fixed 30 epochs
- best source-validation AUROC checkpoint
- no scheduler
- no target tuning

This exactly matches Phase-0A.

---

# Frozen replication gates

## Gate U — UltraFedFM replication

Across the four UltraFedFM−MAE external scenarios, require all:

1. point effect > 0 in at least **3/4** scenarios;
2. median effect ≥ **+0.02 AUROC**;
3. paired bootstrap `P(UltraFedFM > MAE) ≥ 0.90` in at least **3/4** scenarios.

## Gate C — combined two-USFM consistency

Combine:
- 4 Phase-0A URFM−MAE external effects
- 4 Phase-0B UltraFedFM−MAE external effects

Require both:

1. at least **6/8** model×scenario effects are positive;
2. combined median effect ≥ **+0.03 AUROC**.

## Final decision

`GO_FULL_STUDY_TWO_USFM_REPLICATION`

only if Gate U and Gate C both pass.

Otherwise:

`STOP_TWO_USFM_REPLICATION_GATE`

A GO supports investment in the full study; it does **not** remove the UltraFedFM provenance caveat.

A STOP means:
- no label-efficiency matrix;
- no full fine-tuning;
- no LoRA/VPT;
- no third USFM rescue.


In [ ]:
!pip -q install timm==1.0.28 gdown kagglehub pandas scikit-learn pillow tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, gc, json, random, zipfile, shutil, hashlib
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import timm
import gdown
import kagglehub

from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm

ROOT = Path('/content/drive/MyDrive/SPATIOSPECTRAL_PEFT_PILOT')

PHASEA = ROOT/'fm_external_transfer_phase0_v1_0'
PHASEA_OUT = PHASEA/'outputs'
PHASEA_PROG = PHASEA/'progress'
PHASEA_CKPT = PHASEA/'checkpoints'

D1 = ROOT/'linear_readout_control_v1_0'
D1O = D1/'outputs'

PACKA = ROOT/'reviewer_proof_pack_A_v1_0'
PACKAO = PACKA/'outputs'

EXP = ROOT/'ultrafedfm_phase0b_v1_0'
OUT = EXP/'outputs'
PROG = EXP/'progress'
CKPT = EXP/'checkpoints'
MODEL_DIR = ROOT/'external_models'/'UltraFedFM'

for p in [EXP,OUT,PROG,CKPT,MODEL_DIR]:
    p.mkdir(parents=True,exist_ok=True)

# Official Google Drive file linked by the UltraFedFM GitHub README.
ULTRAFED_OFFICIAL_GDRIVE_ID = '13cczqVFk84c_9QDP2OLURU1jWryCky5f'
ULTRAFED_LOCAL = MODEL_DIR/'official_pretrained_ultrafedfm_checkpoint.pth'

REP = 'UltraFedFM-224'
MAE_REP = 'ImageNet-MAE-224'

SOURCE_SPLIT_SEED = 42
HEAD_SEEDS = [42,43,44]
HEAD_BATCH = 16
HEAD_EPOCHS = 30
HEAD_LR = 1e-4
HEAD_WEIGHT_DECAY = 1e-2

FEATURE_BATCH = 8
BOOTSTRAP_REPS = 3000
BOOTSTRAP_SEED = 93001

# Gate U
U_MIN_POSITIVE = 3
U_MEDIAN_EFFECT = 0.02
U_MIN_BOOT_SUPPORTED = 3
U_BOOT_PROB = 0.90

# Gate C
C_MIN_POSITIVE = 6
C_MEDIAN_EFFECT = 0.03

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Feature extraction device:",DEVICE)
if DEVICE.type=='cuda':
    print(torch.cuda.get_device_name(0))

CONFIG={
    'experiment':'UltraFedFM second-USFM external transfer replication v1.0',
    'model':'UltraFedFM ViT-Base pretrained checkpoint',
    'common_feature_protocol':'224x224, ImageNet normalization, CLS token, frozen encoder',
    'generic_baseline':'Phase-0A ImageNet-MAE-224',
    'sources':['BUSBRA','BREAST'],
    'scenarios':[
        'BUSBRA->BREAST','BUSBRA->BUSI',
        'BREAST->BUSBRA','BREAST->BUSI'
    ],
    'head_seeds':HEAD_SEEDS,
    'head_epochs':HEAD_EPOCHS,
    'bootstrap_reps':BOOTSTRAP_REPS,
    'provenance_status':'SUPPORTIVE_REPLICATION_PRETRAINING_MEMBERSHIP_NOT_FULLY_RECONCILED',
    'final_decisions':[
        'GO_FULL_STUDY_TWO_USFM_REPLICATION',
        'STOP_TWO_USFM_REPLICATION_GATE'
    ],
}
(EXP/'FROZEN_PHASE0B_CONFIG.json').write_text(
    json.dumps(CONFIG,indent=2),encoding='utf-8'
)


## 1. Utilities

In [ ]:
IMAGENET_MEAN=torch.tensor([0.485,0.456,0.406],dtype=torch.float32).view(3,1,1)
IMAGENET_STD=torch.tensor([0.229,0.224,0.225],dtype=torch.float32).view(3,1,1)

def seed_everything(seed):
    os.environ['PYTHONHASHSEED']=str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def atomic_csv(df,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    df.to_csv(tmp,index=False)
    os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2),encoding='utf-8')
    os.replace(tmp,path)

def canonical_patient_key(v):
    s=str(v).strip()
    if re.fullmatch(r'\d+(?:\.0+)?',s):
        return str(int(float(s)))
    return s.lower()

def all_exist(values):
    return bool(np.all([Path(str(x)).exists() for x in values]))

def pil_tensor(img):
    img=img.convert('RGB').resize((224,224),Image.Resampling.BILINEAR)
    x=torch.from_numpy(np.asarray(img,dtype=np.float32)/255.0).permute(2,0,1)
    return (x-IMAGENET_MEAN)/IMAGENET_STD

def margin(logits):
    return logits[:,1]-logits[:,0]

def auc_ap(y,s):
    return (
        float(roc_auc_score(y,s)),
        float(average_precision_score(y,s))
    )

def sha256(path,block=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()


## 2. Resolve the official UltraFedFM pretrained checkpoint

The notebook first reuses the persistent Drive copy.

If absent, it downloads the exact Google Drive file linked by the official UltraFedFM repository.

If Google Drive blocks automated download, manually use the official checkpoint link from the
UltraFedFM GitHub README and place the **pretrained** `.pth` file at:

`MyDrive/SPATIOSPECTRAL_PEFT_PILOT/external_models/UltraFedFM/official_pretrained_ultrafedfm_checkpoint.pth`

Do not substitute a downstream-finetuned checkpoint.


In [ ]:
def resolve_ultrafed_checkpoint():
    if ULTRAFED_LOCAL.exists():
        print("Using existing Drive checkpoint:",ULTRAFED_LOCAL)
        return ULTRAFED_LOCAL

    print("Downloading official UltraFedFM Google Drive checkpoint...")
    url=f'https://drive.google.com/uc?id={ULTRAFED_OFFICIAL_GDRIVE_ID}'

    try:
        out=gdown.download(
            url,
            str(ULTRAFED_LOCAL),
            quiet=False,
            fuzzy=True
        )
    except Exception as e:
        out=None
        print("gdown error:",repr(e))

    if out is None or not ULTRAFED_LOCAL.exists():
        raise FileNotFoundError(
            "Automatic download failed. Open the official UltraFedFM GitHub README, "
            "download the pretrained checkpoint from its Google Drive/OneDrive link, "
            "and place it at:\n"+str(ULTRAFED_LOCAL)
        )

    return ULTRAFED_LOCAL

ULTRAFED_CKPT_PATH=resolve_ultrafed_checkpoint()

print("Checkpoint:",ULTRAFED_CKPT_PATH)
print("Size MB:",ULTRAFED_CKPT_PATH.stat().st_size/1e6)
print("SHA256:",sha256(ULTRAFED_CKPT_PATH))


## 3. Load the pretrained UltraFedFM encoder with a strict audit

UltraFedFM's official pretraining checkpoint follows MAE-style encoder/decoder storage.
Only encoder tensors are loaded into a ViT-B/16.

Decoder weights, mask token, and any classifier head are ignored.

Critical encoder keys must load successfully.


In [ ]:
def build_ultrafed():
    ck=torch.load(
        ULTRAFED_CKPT_PATH,
        map_location='cpu',
        weights_only=False
    )

    if not isinstance(ck,dict):
        raise RuntimeError("UltraFedFM checkpoint is not a dictionary.")

    if 'model' in ck:
        state=ck['model']
        state_container='model'
    elif 'state_dict' in ck:
        state=ck['state_dict']
        state_container='state_dict'
    else:
        # Only accept raw state dict when tensor-valued keys look like a model.
        tensor_keys=[k for k,v in ck.items() if torch.is_tensor(v)]
        if len(tensor_keys)<20:
            raise RuntimeError(
                "Could not identify a pretrained model state dict. "
                "This may be the wrong UltraFedFM file."
            )
        state=ck
        state_container='raw'

    cleaned={}
    for k,v in state.items():
        nk=k
        for prefix in ['module.','model.','encoder.']:
            if nk.startswith(prefix):
                nk=nk[len(prefix):]
        cleaned[nk]=v

    model=timm.create_model(
        'vit_base_patch16_224',
        pretrained=False,
        num_classes=0,
        global_pool='token'
    )

    target=model.state_dict()
    filtered={}
    skipped_decoder=[]
    shape_mismatch=[]

    for k,v in cleaned.items():
        if (
            k.startswith('decoder')
            or k=='mask_token'
            or k.startswith('head.')
        ):
            skipped_decoder.append(k)
            continue

        if k in target:
            if tuple(v.shape)==tuple(target[k].shape):
                filtered[k]=v
            else:
                shape_mismatch.append(
                    [k,list(v.shape),list(target[k].shape)]
                )

    msg=model.load_state_dict(filtered,strict=False)

    critical=[
        'patch_embed.proj.weight',
        'cls_token',
        'pos_embed',
        'blocks.0.attn.qkv.weight',
        'blocks.11.attn.qkv.weight',
        'norm.weight',
        'norm.bias',
    ]
    absent=[k for k in critical if k not in filtered]
    if absent:
        raise RuntimeError(
            "Critical UltraFedFM encoder keys were not loaded: "
            +str(absent)
            +"\nThis is likely the wrong checkpoint or an incompatible checkpoint format."
        )

    # Encoder-only missing keys should be absent or classifier-related only.
    unexpected_encoder_missing=[
        k for k in msg.missing_keys
        if not k.startswith('head.')
    ]
    if unexpected_encoder_missing:
        raise RuntimeError(
            "Unexpected missing encoder keys after UltraFedFM load: "
            +str(unexpected_encoder_missing[:20])
        )

    audit={
        'checkpoint_path':str(ULTRAFED_CKPT_PATH),
        'sha256':sha256(ULTRAFED_CKPT_PATH),
        'state_container':state_container,
        'loaded_tensors':len(filtered),
        'critical_keys_loaded':critical,
        'missing_keys':list(msg.missing_keys),
        'unexpected_keys':list(msg.unexpected_keys),
        'shape_mismatch':shape_mismatch,
        'ignored_decoder_or_head_tensors':len(skipped_decoder),
        'feature_protocol':'ViT-B/16 CLS token, 224x224, ImageNet normalization',
        'official_downstream_pooling_note':(
            'UltraFedFM downstream code defaults to global pooling; '
            'this study uses a common CLS-token protocol across pretrained ViT-B encoders.'
        ),
    }
    atomic_json(audit,OUT/'ULTRAFEDFM_LOAD_AUDIT.json')

    for p in model.parameters():
        p.requires_grad=False

    model=model.to(DEVICE).eval()

    with torch.no_grad():
        z=model(torch.zeros(1,3,224,224,device=DEVICE))

    if tuple(z.shape)!=(1,768):
        raise RuntimeError(f"Unexpected UltraFedFM output shape: {tuple(z.shape)}")

    print("PASS UltraFedFM encoder audit.")
    print(json.dumps(audit,indent=2))
    return model

model_ultrafed=build_ultrafed()


## 4. Load and canonicalize the same dataset manifests used in Phase-0A

In [ ]:
busi_candidates=[
    PACKAO/'BUSI_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSI_LINEAR_CONTROL_MANIFEST.csv',
]
busi_path=next((p for p in busi_candidates if p.exists()),None)

busbra_candidates=[
    PACKAO/'BUSBRA_RUNTIME_REMAPPED_MANIFEST.csv',
    D1O/'BUSBRA_EXTERNAL_MANIFEST.csv',
]
busbra_path=next((p for p in busbra_candidates if p.exists()),None)

breast_path=PACKAO/'BREAST_LESION_MANIFEST.csv'

if busi_path is None or busbra_path is None or not breast_path.exists():
    raise FileNotFoundError("Required completed project manifests are missing.")

busi=pd.read_csv(busi_path)
busbra=pd.read_csv(busbra_path)
breast=pd.read_csv(breast_path)

if not all_exist(busi.image_path):
    print("Remapping stale BUSI paths with KaggleHub.")
    root=Path(kagglehub.dataset_download('aryashah2k/breast-ultrasound-images-dataset'))
    idx={}
    for p in root.rglob('*.png'):
        if '_mask' in p.stem.lower(): continue
        idx[(p.parent.name.strip().lower(),p.stem)]=str(p)

    resolved=[]
    missing=[]
    for _,r in busi.iterrows():
        key=(str(r.label_text).strip().lower(),str(r.image_id).strip())
        p=idx.get(key)
        if p is None: missing.append(key)
        resolved.append(p)
    if missing:
        raise RuntimeError(f"BUSI remap failed: {missing[:10]}")
    busi=busi.copy()
    busi['image_path']=resolved

if not all_exist(busbra.image_path):
    candidates=[
        ROOT/'data'/'BUSBRA_OFFICIAL'/'BUSBRA'/'Images',
        ROOT/'data'/'BUS-BRA'/'Images',
    ]
    image_dir=next((p for p in candidates if p.exists()),None)
    if image_dir is None:
        raise FileNotFoundError("BUS-BRA raw image directory not found.")
    remapped=[str(image_dir/f'{str(iid)}.png') for iid in busbra.image_id]
    if not all_exist(remapped):
        raise RuntimeError("BUS-BRA path remap failed.")
    busbra=busbra.copy()
    busbra['image_path']=remapped

if not all_exist(breast.image_path):
    raise FileNotFoundError("BrEaST extracted image paths are stale.")

busi_train=busi[busi.split=='train'].reset_index(drop=True)
busi_val=busi[busi.split=='val'].reset_index(drop=True)
busi_all=pd.concat([busi_train,busi_val],ignore_index=True)

busbra['patient_key']=busbra.patient_key.map(canonical_patient_key)
breast['patient_key']=breast.patient_key.astype(str)

assert len(busi_all)==647
assert len(busbra)==1875
assert busbra.patient_key.nunique()==1064
assert len(breast)==252
assert breast.patient_key.nunique()==252

print("PASS manifests.")


## 5. Interruption-safe UltraFedFM feature extraction

In [ ]:
class ImageRows(Dataset):
    def __init__(self,rows):
        self.rows=rows.reset_index(drop=True)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        r=self.rows.iloc[i]
        img=Image.open(r.image_path).convert('RGB')
        return {
            'x':pil_tensor(img),
            'idx':torch.tensor(i,dtype=torch.long)
        }

def open_mm(path,dtype,shape):
    path=Path(path)
    if not path.exists():
        mm=np.lib.format.open_memmap(path,mode='w+',dtype=dtype,shape=shape)
        mm[:]=0
        mm.flush()
        del mm
    mm=np.load(path,mmap_mode='r+')
    if mm.shape!=shape:
        raise RuntimeError(f'{path.name}: {mm.shape} != {shape}')
    return mm

@torch.no_grad()
def extract(rows,dataset):
    ds=ImageRows(rows)
    n=len(ds)

    feat_path=PROG/f'UltraFedFM_224_{dataset}_features.npy'
    done_path=PROG/f'UltraFedFM_224_{dataset}_done.npy'

    feat=open_mm(feat_path,np.float32,(n,768))
    done=open_mm(done_path,np.bool_,(n,))

    remaining=np.flatnonzero(~np.asarray(done,dtype=bool)).tolist()
    print(dataset,":",n-len(remaining),"/",n,"complete")

    if remaining:
        if DEVICE.type!='cuda':
            raise RuntimeError("GPU required for unfinished UltraFedFM feature extraction.")

        loader=DataLoader(
            Subset(ds,remaining),
            batch_size=FEATURE_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True
        )

        model_ultrafed.eval()

        for b in tqdm(loader,desc=f'UltraFedFM-{dataset}'):
            idx=b['idx'].numpy().astype(int)
            x=b['x'].to(DEVICE,non_blocking=True)
            z=model_ultrafed(x)
            feat[idx]=z.detach().cpu().numpy().astype(np.float32)
            done[idx]=True
            feat.flush()
            done.flush()

    if not bool(np.asarray(done).all()):
        raise RuntimeError(f"Incomplete feature cache for {dataset}")

    out=np.asarray(feat,dtype=np.float32).copy()
    del feat,done
    return out

features_u={
    'BUSI':extract(busi_all,'BUSI'),
    'BUSBRA':extract(busbra,'BUSBRA'),
    'BREAST':extract(breast,'BREAST'),
}

model_ultrafed.to('cpu')
del model_ultrafed
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

assert features_u['BUSI'].shape==(647,768)
assert features_u['BUSBRA'].shape==(1875,768)
assert features_u['BREAST'].shape==(252,768)

print("PASS UltraFedFM feature caches.")


## 6. Reuse Phase-0A ImageNet-MAE features and exact source splits

No MAE image inference or head training is repeated.


In [ ]:
mae_feature_paths={
    'BUSI':PHASEA_PROG/'ImageNet_MAE_224_BUSI_features.npy',
    'BUSBRA':PHASEA_PROG/'ImageNet_MAE_224_BUSBRA_features.npy',
    'BREAST':PHASEA_PROG/'ImageNet_MAE_224_BREAST_features.npy',
}

missing={k:str(v) for k,v in mae_feature_paths.items() if not v.exists()}
if missing:
    raise FileNotFoundError(
        "Phase-0A MAE feature caches missing; do not recompute automatically:\n"
        +json.dumps(missing,indent=2)
    )

features_m={
    k:np.asarray(np.load(v,mmap_mode='r'),dtype=np.float32)
    for k,v in mae_feature_paths.items()
}

split_paths={
    'BUSBRA':PHASEA_OUT/'BUSBRA_PHASE0_SOURCE_SPLIT.csv',
    'BREAST':PHASEA_OUT/'BREAST_PHASE0_SOURCE_SPLIT.csv',
}
for p in split_paths.values():
    if not p.exists():
        raise FileNotFoundError(p)

saved_splits={
    k:pd.read_csv(v)
    for k,v in split_paths.items()
}

print("PASS Phase-0A MAE caches and source split artifacts.")


## 7. Build patient/case source units and recover exact source split indices

In [ ]:
def patient_mean_features(X,manifest):
    groups=manifest.groupby(['patient_key','y'],sort=True).indices
    rows=[]
    vecs=[]
    for (pid,y),idx in groups.items():
        idx=np.asarray(idx,dtype=int)
        rows.append({
            'patient_key':str(pid),
            'y':int(y),
            'n_images':len(idx)
        })
        vecs.append(np.asarray(X[idx],dtype=np.float32).mean(axis=0))
    return pd.DataFrame(rows),np.stack(vecs).astype(np.float32)

source_units={}

for rep_name,feat in [('UltraFedFM',features_u),('MAE',features_m)]:
    meta_bra,X_bra=patient_mean_features(feat['BUSBRA'],busbra)
    source_units[(rep_name,'BUSBRA')]={'meta':meta_bra,'X':X_bra,'y':meta_bra.y.to_numpy(int)}

    meta_br=breast[['patient_key','y']].copy().reset_index(drop=True)
    source_units[(rep_name,'BREAST')]={'meta':meta_br,'X':feat['BREAST'],'y':meta_br.y.to_numpy(int)}

splits={}

for source in ['BUSBRA','BREAST']:
    meta=source_units[('UltraFedFM',source)]['meta'].copy()
    saved=saved_splits[source].copy()
    saved['patient_key']=saved.patient_key.astype(str)
    split_map=dict(zip(saved.patient_key,saved.source_split))

    labels=meta.patient_key.astype(str).map(split_map)
    if labels.isna().any():
        raise RuntimeError(f"{source}: could not recover exact Phase-0A split for all units.")

    tr=np.flatnonzero(labels.to_numpy()=='train')
    va=np.flatnonzero(labels.to_numpy()=='val')
    splits[source]=(tr,va)

    assert set(meta.iloc[tr].patient_key).isdisjoint(set(meta.iloc[va].patient_key))

    # Representation identities must match.
    m2=source_units[('MAE',source)]['meta']
    assert np.array_equal(meta.patient_key.astype(str).to_numpy(),m2.patient_key.astype(str).to_numpy())
    assert np.array_equal(meta.y.to_numpy(int),m2.y.to_numpy(int))

    print(source,"train",len(tr),"val",len(va))


## 8. Train UltraFedFM linear heads only

In [ ]:
class FeatDS(Dataset):
    def __init__(self,X,y):
        self.X=np.asarray(X,dtype=np.float32)
        self.y=np.asarray(y,dtype=np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return torch.from_numpy(self.X[i]),torch.tensor(int(self.y[i]),dtype=torch.long)

def weights_for(y):
    c=np.bincount(y,minlength=2)
    w=c.sum()/(2*np.maximum(c,1))
    w=w/w.mean()
    return torch.tensor(w,dtype=torch.float32)

@torch.no_grad()
def head_scores(head,X):
    head.eval()
    loader=DataLoader(
        torch.from_numpy(np.asarray(X,dtype=np.float32)),
        batch_size=256,shuffle=False,num_workers=0
    )
    scores=[]
    for x in loader:
        scores.extend(margin(head(x)).numpy().tolist())
    return np.asarray(scores,dtype=float)

def train_ultrafed_head(source,seed):
    seed_everything(seed)

    tag=f'UltraFedFM_224_{source}_seed{seed}'
    best_path=CKPT/f'{tag}_best.pt'
    hist_path=OUT/f'{tag}_history.csv'
    status_path=OUT/f'{tag}_status.json'

    if best_path.exists() and status_path.exists():
        st=json.loads(status_path.read_text(encoding='utf-8'))
        if st.get('complete_30_epochs',False):
            h=nn.Linear(768,2)
            ck=torch.load(best_path,map_location='cpu',weights_only=False)
            h.load_state_dict(ck['state_dict'])
            h.eval()
            return h,st,pd.read_csv(hist_path)

    X=source_units[('UltraFedFM',source)]['X']
    y=source_units[('UltraFedFM',source)]['y']
    tr,va=splits[source]

    gen=torch.Generator()
    gen.manual_seed(seed)

    loader=DataLoader(
        FeatDS(X[tr],y[tr]),
        batch_size=HEAD_BATCH,
        shuffle=True,
        num_workers=0,
        generator=gen
    )

    h=nn.Linear(768,2)
    opt=torch.optim.AdamW(
        h.parameters(),
        lr=HEAD_LR,
        weight_decay=HEAD_WEIGHT_DECAY
    )
    cw=weights_for(y[tr])

    best_auc=-np.inf
    best_epoch=None
    hist=[]

    for epoch in range(1,HEAD_EPOCHS+1):
        h.train()
        losses=[]
        for f,yy in loader:
            opt.zero_grad(set_to_none=True)
            logits=h(f)
            loss=F.cross_entropy(logits,yy,weight=cw)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        s=head_scores(h,X[va])
        auc,ap=auc_ap(y[va],s)

        hist.append({
            'representation':'UltraFedFM-224',
            'source':source,
            'seed':seed,
            'epoch':epoch,
            'train_loss':float(np.mean(losses)),
            'source_val_auc':auc,
            'source_val_auprc':ap
        })

        if auc>best_auc:
            best_auc=auc
            best_epoch=epoch
            torch.save({
                'state_dict':{k:v.detach().cpu().clone() for k,v in h.state_dict().items()},
                'epoch':epoch,
                'best_source_val_auc':best_auc,
                'source':source,
                'seed':seed
            },best_path)

    hist=pd.DataFrame(hist)
    atomic_csv(hist,hist_path)

    ck=torch.load(best_path,map_location='cpu',weights_only=False)
    h.load_state_dict(ck['state_dict'])
    h.eval()

    st={
        'representation':'UltraFedFM-224',
        'source':source,
        'seed':seed,
        'complete_30_epochs':True,
        'best_epoch':int(best_epoch),
        'best_source_val_auc':float(best_auc),
        'trainable_params':1538
    }
    atomic_json(st,status_path)
    return h,st,hist

heads_u={}
status_rows=[]

for source in ['BUSBRA','BREAST']:
    for seed in HEAD_SEEDS:
        h,st,hist=train_ultrafed_head(source,seed)
        heads_u[(source,seed)]=h
        status_rows.append(st)

status=pd.DataFrame(status_rows)
atomic_csv(status,OUT/'ULTRAFEDFM_HEAD_TRAINING_STATUS.csv')
display(status.round(4))


## 9. Load the already-trained Phase-0A MAE heads

In [ ]:
heads_m={}

for source in ['BUSBRA','BREAST']:
    for seed in HEAD_SEEDS:
        p=PHASEA_CKPT/f'ImageNet_MAE_224_{source}_seed{seed}_best.pt'
        if not p.exists():
            raise FileNotFoundError(
                "Phase-0A MAE head checkpoint missing: "+str(p)
            )
        ck=torch.load(p,map_location='cpu',weights_only=False)
        h=nn.Linear(768,2)
        h.load_state_dict(ck['state_dict'])
        h.eval()
        heads_m[(source,seed)]=h

print("PASS all six Phase-0A MAE heads reused.")


## 10. Source-validation comparison

In [ ]:
source_rows=[]

for source in ['BUSBRA','BREAST']:
    tr,va=splits[source]

    for seed in HEAD_SEEDS:
        for model_name,heads,rep_name in [
            ('UltraFedFM',heads_u,'UltraFedFM'),
            ('ImageNet-MAE',heads_m,'MAE')
        ]:
            X=source_units[(rep_name,source)]['X']
            y=source_units[(rep_name,source)]['y']
            s=head_scores(heads[(source,seed)],X[va])
            auc,ap=auc_ap(y[va],s)

            source_rows.append({
                'model':model_name,
                'source':source,
                'seed':seed,
                'auc':auc,
                'auprc':ap,
                'n_val':len(va)
            })

source_result=pd.DataFrame(source_rows)
atomic_csv(source_result,OUT/'ULTRAFED_VS_MAE_SOURCE_VALIDATION.csv')
display(source_result.round(4))


## 11. External scores for UltraFedFM and reused MAE

In [ ]:
def busbra_patient_scores(img_scores):
    t=busbra[['patient_key','y']].copy().reset_index(drop=True)
    t['score']=np.asarray(img_scores,dtype=float)
    return (
        t.groupby(['patient_key','y'],as_index=False)
        .agg(score=('score','mean'))
        .sort_values('patient_key')
        .reset_index(drop=True)
    )

scenarios=[
    ('BUSBRA','BREAST'),
    ('BUSBRA','BUSI'),
    ('BREAST','BUSBRA'),
    ('BREAST','BUSI')
]

scores={}
rows=[]

for source,target in scenarios:
    for model_name,heads,feat in [
        ('UltraFedFM',heads_u,features_u),
        ('ImageNet-MAE',heads_m,features_m),
    ]:
        for seed in HEAD_SEEDS:
            h=heads[(source,seed)]

            if target=='BUSBRA':
                raw=head_scores(h,feat['BUSBRA'])
                p=busbra_patient_scores(raw)
                y=p.y.to_numpy(int)
                s=p.score.to_numpy(float)
                ids=p.patient_key.astype(str).to_numpy()
            elif target=='BREAST':
                y=breast.y.to_numpy(int)
                s=head_scores(h,feat['BREAST'])
                ids=breast.patient_key.astype(str).to_numpy()
            else:
                y=busi_all.y.to_numpy(int)
                s=head_scores(h,feat['BUSI'])
                ids=busi_all.image_id.astype(str).to_numpy()

            scores[(model_name,source,target,seed)]={
                'ids':ids,'y':y,'score':s
            }

            auc,ap=auc_ap(y,s)
            rows.append({
                'model':model_name,
                'source':source,
                'target':target,
                'seed':seed,
                'n':len(y),
                'auc':auc,
                'auprc':ap
            })

external=pd.DataFrame(rows)
atomic_csv(external,OUT/'ULTRAFED_VS_MAE_EXTERNAL_RESULTS.csv')
display(external.round(4))


## 12. Paired pathology-stratified bootstrap

Primary Phase-0B effect:

\[
\Delta =
\frac{1}{3}\sum_s
(AUC_{UltraFedFM,s}-AUC_{MAE,s})
\]

The same resampled target units are used for both models and all seeds.


In [ ]:
effect_rows=[]
boot_rows=[]

for si,(source,target) in enumerate(scenarios):
    ref=scores[('UltraFedFM',source,target,HEAD_SEEDS[0])]
    y=ref['y']
    ids=ref['ids']

    for model_name in ['UltraFedFM','ImageNet-MAE']:
        for seed in HEAD_SEEDS:
            cur=scores[(model_name,source,target,seed)]
            assert np.array_equal(ids,cur['ids'])
            assert np.array_equal(y,cur['y'])

    seed_deltas=[]
    for seed in HEAD_SEEDS:
        u=scores[('UltraFedFM',source,target,seed)]['score']
        m=scores[('ImageNet-MAE',source,target,seed)]['score']
        seed_deltas.append(
            float(roc_auc_score(y,u)-roc_auc_score(y,m))
        )

    point=float(np.mean(seed_deltas))

    i0=np.flatnonzero(y==0)
    i1=np.flatnonzero(y==1)

    rng=np.random.default_rng(BOOTSTRAP_SEED+si)
    ds=[]

    for b in range(BOOTSTRAP_REPS):
        s0=rng.choice(i0,size=len(i0),replace=True)
        s1=rng.choice(i1,size=len(i1),replace=True)
        idx=np.concatenate([s0,s1])
        yy=y[idx]

        per_seed=[]
        for seed in HEAD_SEEDS:
            u=scores[('UltraFedFM',source,target,seed)]['score'][idx]
            m=scores[('ImageNet-MAE',source,target,seed)]['score'][idx]
            per_seed.append(
                roc_auc_score(yy,u)-roc_auc_score(yy,m)
            )
        ds.append(float(np.mean(per_seed)))

    ds=np.asarray(ds,dtype=float)

    effect_rows.append({
        'source':source,
        'target':target,
        'ultrafed_minus_mae_auc_point':point,
        'seed42_delta':seed_deltas[0],
        'seed43_delta':seed_deltas[1],
        'seed44_delta':seed_deltas[2],
        'bootstrap_median':float(np.median(ds)),
        'ci_lo':float(np.quantile(ds,.025)),
        'ci_hi':float(np.quantile(ds,.975)),
        'p_ultrafed_better':float(np.mean(ds>0)),
    })

    boot_rows.append(pd.DataFrame({
        'source':source,
        'target':target,
        'bootstrap_rep':np.arange(BOOTSTRAP_REPS),
        'ultrafed_minus_mae_auc':ds
    }))

effects=pd.DataFrame(effect_rows)
boot=pd.concat(boot_rows,ignore_index=True)

atomic_csv(effects,OUT/'PRIMARY_ULTRAFED_VS_MAE_EXTERNAL_EFFECTS.csv')
atomic_csv(boot,OUT/'PRIMARY_ULTRAFED_VS_MAE_BOOTSTRAP_3000.csv')

display(effects.round(4))


## 13. Combine with frozen Phase-0A URFM results and apply final gate

In [ ]:
phasea_effect_path=PHASEA_OUT/'PRIMARY_URFM_VS_MAE_EXTERNAL_EFFECTS.csv'
if not phasea_effect_path.exists():
    raise FileNotFoundError(phasea_effect_path)

urfm=pd.read_csv(phasea_effect_path)

# Gate U
u_positive=int((effects.ultrafed_minus_mae_auc_point>0).sum())
u_median=float(effects.ultrafed_minus_mae_auc_point.median())
u_boot=int((effects.p_ultrafed_better>=U_BOOT_PROB).sum())

gate_u=(
    u_positive>=U_MIN_POSITIVE
    and u_median>=U_MEDIAN_EFFECT
    and u_boot>=U_MIN_BOOT_SUPPORTED
)

# Combined 8 model x scenario effects.
combined_rows=[]

for _,r in urfm.iterrows():
    combined_rows.append({
        'ultrasound_fm':'URFM',
        'source':r.source,
        'target':r.target,
        'fm_minus_mae_auc':float(r.urfm_minus_mae_auc_point)
    })

for _,r in effects.iterrows():
    combined_rows.append({
        'ultrasound_fm':'UltraFedFM',
        'source':r.source,
        'target':r.target,
        'fm_minus_mae_auc':float(r.ultrafed_minus_mae_auc_point)
    })

combined=pd.DataFrame(combined_rows)

c_positive=int((combined.fm_minus_mae_auc>0).sum())
c_median=float(combined.fm_minus_mae_auc.median())

gate_c=(
    c_positive>=C_MIN_POSITIVE
    and c_median>=C_MEDIAN_EFFECT
)

decision=(
    'GO_FULL_STUDY_TWO_USFM_REPLICATION'
    if gate_u and gate_c
    else 'STOP_TWO_USFM_REPLICATION_GATE'
)

decision_obj={
    'decision':decision,
    'gate_U_ultrafed_replication':{
        'pass':bool(gate_u),
        'positive_scenarios':u_positive,
        'required_positive_scenarios':U_MIN_POSITIVE,
        'median_ultrafed_minus_mae_auc':u_median,
        'required_median_effect':U_MEDIAN_EFFECT,
        'bootstrap_supported_scenarios':u_boot,
        'required_bootstrap_supported_scenarios':U_MIN_BOOT_SUPPORTED,
    },
    'gate_C_combined_two_usfm':{
        'pass':bool(gate_c),
        'positive_model_scenarios':c_positive,
        'required_positive_model_scenarios':C_MIN_POSITIVE,
        'combined_median_fm_minus_mae_auc':c_median,
        'required_combined_median_effect':C_MEDIAN_EFFECT,
    },
    'provenance_status':(
        'UltraFedFM is supportive replication. The main article uses BUSI/BUS-BRA/BUS-UCLM '
        'as downstream/fairness datasets; exact public pretraining membership must be reconciled '
        'from Supplementary Table 1 before any zero-overlap claim.'
    ),
    'post_decision_rule':(
        'GO permits design of the full study. STOP means no label-efficiency, full-finetuning, '
        'PEFT, third-USFM, or other rescue experiment in this breast-US direction.'
    )
}

atomic_csv(combined,OUT/'TWO_USFM_COMBINED_EXTERNAL_EFFECTS.csv')
atomic_json(decision_obj,OUT/'PHASE0B_FINAL_DECISION.json')

display(combined.round(4))
print(json.dumps(decision_obj,indent=2))


## 14. Package results

Return:

`ULTRAFEDFM_PHASE0B_RESULTS.zip`

Do not begin the full study until this ZIP has been reviewed.


In [ ]:
result_zip=ROOT/'ULTRAFEDFM_PHASE0B_RESULTS.zip'

files_to_pack=[
    EXP/'FROZEN_PHASE0B_CONFIG.json',
    OUT/'ULTRAFEDFM_LOAD_AUDIT.json',
    OUT/'ULTRAFEDFM_HEAD_TRAINING_STATUS.csv',
    OUT/'ULTRAFED_VS_MAE_SOURCE_VALIDATION.csv',
    OUT/'ULTRAFED_VS_MAE_EXTERNAL_RESULTS.csv',
    OUT/'PRIMARY_ULTRAFED_VS_MAE_EXTERNAL_EFFECTS.csv',
    OUT/'PRIMARY_ULTRAFED_VS_MAE_BOOTSTRAP_3000.csv',
    OUT/'TWO_USFM_COMBINED_EXTERNAL_EFFECTS.csv',
    OUT/'PHASE0B_FINAL_DECISION.json',
]

for source in ['BUSBRA','BREAST']:
    for seed in HEAD_SEEDS:
        tag=f'UltraFedFM_224_{source}_seed{seed}'
        files_to_pack += [
            OUT/f'{tag}_history.csv',
            OUT/f'{tag}_status.json'
        ]

with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in files_to_pack:
        if not p.exists():
            raise FileNotFoundError(p)
        z.write(p,arcname=p.name)

print("Created:",result_zip)
print("Size MB:",result_zip.stat().st_size/1e6)
